# PROC COMPUTAB による四半期プロフォーマ損益計算書

## エグゼクティブサマリー

このノートブックでは、SAS/ETS のレポート作成用テーブルプロシージャである PROC COMPUTAB を用いて、月次の元帳データから地方銀行の四半期プロフォーマ損益計算書を直接作成します。各月の受取利息、支払利息、手数料収入、営業費用を該当する暦四半期の列へ振り分け、続いて行プログラミングブロックで純受取利息、税引前利益、当期純利益を計算し、列ブロックで4つの四半期を通年合計に集計します。

## データソース

単一の合成データセット `bank_ledger` は、中規模のコミュニティ銀行における1会計年度分の月次財務諸表明細をシミュレートします。12か月分の月次オブザベーションは `call streaminit`/`rand` を使ってインラインで生成されるため、このノートブックは完全に自己完結しています。

| 変数 | 型 | 説明 |
|----------|------|-------------|
| `stmtdate` | 数値 (DATE9.) | 月末決算日（1月〜12月） |
| `loanint`  | 数値 | 貸出ポートフォリオで得た利息および手数料（USD千単位） |
| `secint`   | 数値 | 投資有価証券で得た利息（USD千単位） |
| `depint`   | 数値 | 預金に支払った利息（USD千単位） |
| `borrint`  | 数値 | 借入金 / FHLB アドバンスに支払った利息（USD千単位） |
| `feeinc`   | 数値 | 非金利（手数料・サービス料）収入（USD千単位） |
| `salaries` | 数値 | 給与および従業員福利厚生費（USD千単位） |
| `occupancy`| 数値 | 施設費および設備費（USD千単位） |
| `othropex` | 数値 | その他の非金利営業費用（USD千単位） |
| `provision`| 数値 | 貸倒引当金繰入額（USD千単位） |
| `taxrate`  | 数値 | 税引前利益に適用する実効税率 |

# PROC COMPUTAB による四半期プロフォーマ損益計算書

銀行の財務チームは、月次総勘定元帳を、純受取利息と最終的な当期純利益を示す**四半期損益計算書**へと日常的に変換しています。`PROC COMPUTAB`（SAS/ETS）はまさにこの目的のために作られており、**列**を報告期間、**行**を財務諸表の明細項目とするプログラム可能なテーブルを構成し、通常の DATA ステップのロジックを行ブロックと列ブロックの中で用いて小計を計算できます。

このノートブックでは次のことを行います。

1. コミュニティ銀行の1会計年度分の合成月次元帳データを生成する。
2. 各月をその暦四半期に振り分け、四半期損益計算書を作成する。
3. **行ブロック**で純受取利息、税引前利益、当期純利益を計算し、**列ブロック**で各四半期を通年合計に集計する。
4. `out=` テーブルを再利用し、計算済みの財務諸表を下流のレポートに供給できるようにする。

## ステップ1 — 合成月次元帳データの生成

12か月分の月末オブザベーションをシミュレートします。貸出および有価証券の受取利息は年間を通じて緩やかに上昇し、預金および借入のコストは金利環境に応じて変動し、手数料収入、営業費用、貸倒引当金には現実的な月次のばらつきを持たせます。`call streaminit` はシードを固定するため、データは再現可能です。

In [1]:
データ bank_ledger;
   呼出 streaminit(20240115);
   書式 stmtdate date9.;
   繰返 month = 1 から 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      出力;
   終了;
   保持 stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
実行;

処理 印刷 データ=bank_ledger noobs 見出;
   表題 'Synthetic Monthly Ledger — Community Bank FY2024';
   書式 loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
実行;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## ステップ2 — 四半期損益計算書の作成

このレポートの中核は、下記の `PROC COMPUTAB` ステップです。

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** は4つの四半期列に加えて通年列を定義します。
- ラベルのない**入力ブロック**は自動変数 **`_col_`** に `qtr(stmtdate)` を設定し、各月次オブザベーションを正しい四半期列へと振り分けます。入力はデフォルトで転置されるため、各元帳変数は同名の行に配置されます。
- **行ブロック** `inc_stmt:` は各列につき1回実行され、純受取利息、非金利費用合計、税引前利益、当期純利益といった導出項目を、会計士が行うのとまったく同じように計算します。
- **列ブロック** `total:` は各行につき1回実行され、4つの四半期を合計して `fy`（通年）列にまとめます。

`rows ... / ...` ステートメントはタイトル、インデント、罫線（`ol` 上罫線、`ul` 下罫線、`dul` 二重下罫線）を追加し、出力が実際の財務諸表のように読めるようにします。

In [2]:
表題 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

処理 computab データ=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / 書式=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
実行;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## ステップ3 — COMPUTAB 出力データセットの再利用

上記の `PROC COMPUTAB` ステップは、`out=` によって計算済みテーブルを `qtr_income` に書き出しました。このデータセットの各行は財務諸表の明細項目であり、各列変数（`qtr1`〜`qtr4`、`fy`）が計算された値を保持しているため、下流のレポートに供給できます。以下では、集計された通年列を印刷して数値が整合していることを確認します。

In [3]:
表題 'COMPUTAB Output Dataset — Full-Year Column';

処理 印刷 データ=qtr_income 見出 noobs;
   変数 _row_ fy;
   書式 fy comma12.1;
   見出 _row_ = 'Line Item' fy = 'Full Year';
実行;

表題;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## 結果の解釈

このプロフォーマ計算書は、銀行の規制上のコールレポートのように上から下へと読めます。受取利息合計から支払利息合計を差し引くと**純受取利息**（ここでは年間で約 \$20.5 百万）が得られ、これが銀行の主たる収益源となります。そこから**貸倒引当金**を差し引き、**手数料収入**を加え、**非金利費用**を控除すると税引前利益は約 \$9.0 百万となり、21% の実効税率を適用すると**当期純利益**は約 \$7.1 百万となります。`total:` 列ブロックが4つの四半期を通年列に合計するため、年間合計は構成上必ず各四半期の合計と一致します。これは `out=` データセットで確認でき、通年の `netinc` 7,081.6 は4つの四半期の値の合計に等しくなっています。

すべてが `PROC COMPUTAB` のプログラム可能な行ブロックと列ブロックの内部で計算されるため、実際の月次元帳に差し替えてもレポートのロジックを変更する必要はなく、入力データセットが変わるだけです。`out=` データセットにより、計算済みの財務諸表をダッシュボード、トレンド分析、あるいは取締役会資料の自動作成に利用できるようになります。